# Demo 04 : History and time travel

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · state-history

We port ARI510 demo22's checkpoint history and fork. A draft node calls the model, then a record node copies the draft into state without an external action.

In [2]:
import csv
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

with open(data("demo-world/tickets.csv"), newline="", encoding="utf-8") as file:
    ticket = next(csv.DictReader(file))

class Reply(TypedDict):
    symptom: str
    draft: str
    recorded: str

calls = []
def draft(state: Reply):
    calls.append("draft")
    return {"draft": llm.invoke("Write one short acknowledgement of this bus service report. Do not claim a repair is complete: " + state["symptom"]).content}

builder = StateGraph(Reply)
builder.add_node("draft", draft)
builder.add_node("record", lambda state: {"recorded": state["draft"]})
builder.add_edge(START, "draft")
builder.add_edge("draft", "record")
builder.add_edge("record", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "ticket-history"}}
original = graph.invoke({"symptom": ticket["symptom"], "draft": "", "recorded": ""}, config)
history = list(graph.get_state_history(config))
for snapshot in history:
    print("step :", snapshot.metadata["step"], "next :", snapshot.next, "draft :", snapshot.values.get("draft", ""))
before_draft = next(snapshot for snapshot in history if snapshot.next == ("draft",))
before_record = next(snapshot for snapshot in history if snapshot.next == ("record",))
original_final = graph.get_state(config).config
print("original :", original["recorded"])

step : 2 next : () draft : Acknowledged—the inverter communication dropout reported by drivers has been noted for investigation.
step : 1 next : ('record',) draft : Acknowledged—the inverter communication dropout reported by drivers has been noted for investigation.
step : 0 next : ('draft',) draft : 
step : -1 next : ('__start__',) draft : 
original : Acknowledged—the inverter communication dropout reported by drivers has been noted for investigation.


## b · replay

We invoke None with a historical checkpoint config. Execution continues from its next nodes, so replay before draft calls the model again and can change the wording.

In [3]:
count_before = len(calls)
replayed = graph.invoke(None, before_draft.config)
print("replayed :", replayed["recorded"])
print("new model calls :", len(calls) - count_before)
print("saved original :", graph.get_state(original_final).values["recorded"])

replayed : Acknowledged—drivers have reported an inverter communication dropout.
new model calls : 1
saved original : Acknowledged—the inverter communication dropout reported by drivers has been noted for investigation.


## c · forking

We edit the saved draft as an update from the draft node. The returned config selects a new checkpoint whose next node is record.

In [4]:
forked = graph.update_state(before_record.config, {"draft": "Bus report received. The workshop will review it."}, as_node="draft")
count_before = len(calls)
alternative = graph.invoke(None, forked)
print("fork :", alternative["recorded"])
print("new model calls :", len(calls) - count_before)
print("parent unchanged :", graph.get_state(before_record.config).values["draft"] == before_record.values["draft"])

fork : Bus report received. The workshop will review it.
new model calls : 0
parent unchanged : True


We also inspect DeltaChannel on this pin. Its reducer receives a batch of writes, and the checkpoint saver reconstructs the accumulated list from saved writes. The API is beta.

In [5]:
from typing import Annotated
from langgraph.channels import DeltaChannel

def merge_batches(current, writes):
    return current + [item for batch in writes for item in batch]

class LongThread(TypedDict):
    entries: Annotated[list[str], DeltaChannel(merge_batches, snapshot_frequency=2)]

long_builder = StateGraph(LongThread)
long_builder.add_node("append", lambda state: {"entries": ["workshop checked"]})
long_builder.add_edge(START, "append")
long_builder.add_edge("append", END)
long_graph = long_builder.compile(checkpointer=InMemorySaver())
long_config = {"configurable": {"thread_id": "delta-thread"}}
long_graph.invoke({"entries": ["ticket opened"]}, long_config)
long_graph.invoke({"entries": ["parts arrived"]}, long_config)
print("delta restored :", long_graph.get_state(long_config).values["entries"])
print("delta history :", [snapshot.values["entries"] for snapshot in long_graph.get_state_history(long_config) if "entries" in snapshot.values])

delta restored : ['ticket opened', 'workshop checked', 'parts arrived', 'workshop checked']
delta history : [['ticket opened', 'workshop checked', 'parts arrived', 'workshop checked'], ['ticket opened', 'workshop checked', 'parts arrived'], ['ticket opened', 'workshop checked'], ['ticket opened', 'workshop checked'], ['ticket opened'], []]


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.